# eQMARL-SLAM benchmark analysis

This notebook reads **schema version 3** result files produced by `SLAM.run_benchmark`. It does not load the incompatible legacy weights or legacy three-channel results.

Run a benchmark first, for example:

```bash
python -m SLAM.run_benchmark --quick --frameworks sctde fctde
```


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)


In [ ]:
def locate_result_file():
    candidates = []
    for root in (Path("results"), Path("SLAM/results")):
        latest = root / "latest.json"
        if latest.exists():
            return latest
        candidates.extend(root.glob("slam-benchmark-*.json"))
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

result_file = locate_result_file()
if result_file is None:
    results = None
    print("No benchmark JSON found. Run SLAM.run_benchmark first.")
else:
    results = json.loads(result_file.read_text(encoding="utf-8"))
    if results.get("schema_version") != 3:
        raise ValueError(
            f"Expected schema_version=3, found {results.get('schema_version')}"
        )
    print(f"Loaded: {result_file.resolve()}")
    print(f"Environment: {results['config']['env_id']}")
    print(f"Completed: {results.get('completed_at_utc', 'in progress')}")


## Seed-level held-out evaluation

Each row below is one independently initialized model seed evaluated on the same held-out map seed set. Statistical comparisons should use these seed-level rows, not individual adjacent training episodes.


In [ ]:
def completed_run_rows(payload):
    rows = []
    if payload is None:
        return rows
    for framework, framework_runs in payload.get("runs", {}).items():
        for seed, run in framework_runs.items():
            if run.get("status") != "completed":
                continue
            summary = run["evaluation"]["summary"]
            rows.append({
                "framework": framework,
                "model_seed": int(seed),
                "eval_return": summary.get("mean_episode_return", np.nan),
                "final_coverage": summary.get("mean_final_coverage", np.nan),
                "coverage_auc": summary.get("mean_coverage_auc", np.nan),
                "success_rate": summary.get("success_rate", np.nan),
                "reach_50_rate": summary.get("reach_50_rate", np.nan),
                "reach_75_rate": summary.get("reach_75_rate", np.nan),
                "reach_90_rate": summary.get("reach_90_rate", np.nan),
                "episode_length": summary.get("mean_episode_length", np.nan),
                "collision_rate": summary.get("mean_collision_rate", np.nan),
                "actor_parameters": run["parameters"]["actor"],
                "critic_parameters": run["parameters"]["critic"],
                "total_parameters": run["parameters"]["total"],
            })
    return rows

seed_df = pd.DataFrame(completed_run_rows(results))
if seed_df.empty:
    print("No completed runs in this result file.")
else:
    display(seed_df.sort_values(["framework", "model_seed"]).reset_index(drop=True))


In [ ]:
if not seed_df.empty:
    aggregate_columns = [
        "eval_return",
        "final_coverage",
        "coverage_auc",
        "success_rate",
        "reach_50_rate",
        "reach_75_rate",
        "reach_90_rate",
        "episode_length",
        "collision_rate",
        "total_parameters",
    ]
    aggregate_df = seed_df.groupby("framework")[aggregate_columns].agg(["mean", "std"])
    display(aggregate_df)


## Training return curves across model seeds

In [ ]:
def rolling_mean(values, window=50):
    return pd.Series(values, dtype=float).rolling(window, min_periods=1).mean().to_numpy()

if results is not None:
    for framework, framework_runs in results.get("runs", {}).items():
        histories = []
        for run in framework_runs.values():
            if run.get("status") != "completed":
                continue
            values = [episode["episode_return"] for episode in run["train"]["episodes"]]
            histories.append(rolling_mean(values))
        if not histories:
            continue
        common_length = min(map(len, histories))
        matrix = np.stack([history[:common_length] for history in histories])
        mean = matrix.mean(axis=0)
        std = matrix.std(axis=0, ddof=1) if len(histories) > 1 else np.zeros(common_length)
        x = np.arange(common_length)
        plt.figure(figsize=(10, 4.5))
        plt.plot(x, mean, label=f"{framework} mean")
        plt.fill_between(x, mean - std, mean + std, alpha=0.2, label="±1 seed SD")
        plt.xlabel("Training episode")
        plt.ylabel("Rolling mean team return")
        plt.title(f"{framework}: training return")
        plt.legend()
        plt.tight_layout()
        plt.show()


## Held-out coverage and success

In [ ]:
if not seed_df.empty:
    evaluation_means = seed_df.groupby("framework")[[
        "final_coverage", "coverage_auc", "success_rate"
    ]].mean()
    for metric, label in (
        ("final_coverage", "Mean final coverage"),
        ("coverage_auc", "Mean fixed-horizon coverage AUC"),
        ("success_rate", "Mean success rate"),
    ):
        plt.figure(figsize=(7, 4.5))
        evaluation_means[metric].sort_values(ascending=False).plot(kind="bar")
        plt.ylabel(label)
        plt.ylim(0.0, 1.0)
        plt.title(label + " on held-out maps")
        plt.tight_layout()
        plt.show()


## Parameter efficiency

In [ ]:
if not seed_df.empty:
    parameter_df = seed_df.groupby("framework").agg(
        total_parameters=("total_parameters", "first"),
        mean_eval_return=("eval_return", "mean"),
        mean_coverage_auc=("coverage_auc", "mean"),
        mean_success_rate=("success_rate", "mean"),
    ).sort_values("total_parameters")
    parameter_df["coverage_auc_per_million_parameters"] = (
        parameter_df["mean_coverage_auc"]
        / (parameter_df["total_parameters"] / 1_000_000.0)
    )
    display(parameter_df)

    plt.figure(figsize=(7, 4.5))
    plt.scatter(parameter_df["total_parameters"], parameter_df["mean_coverage_auc"])
    for framework, row in parameter_df.iterrows():
        plt.annotate(framework, (row["total_parameters"], row["mean_coverage_auc"]))
    plt.xlabel("Total trainable parameters")
    plt.ylabel("Mean coverage AUC")
    plt.title("Held-out coverage AUC vs. parameter count")
    plt.tight_layout()
    plt.show()


## Optimization diagnostics

In [ ]:
if results is not None:
    diagnostic_keys = (
        "actor_loss",
        "critic_loss",
        "mean_policy_entropy",
        "actor_grad_norm",
        "critic_grad_norm",
    )
    for framework, framework_runs in results.get("runs", {}).items():
        completed = [run for run in framework_runs.values() if run.get("status") == "completed"]
        if not completed:
            continue
        for key in diagnostic_keys:
            histories = []
            for run in completed:
                values = [
                    episode.get(key, np.nan)
                    for episode in run["train"]["episodes"]
                ]
                if np.isfinite(np.asarray(values, dtype=float)).any():
                    histories.append(rolling_mean(values))
            if not histories:
                continue
            common_length = min(map(len, histories))
            matrix = np.stack([history[:common_length] for history in histories])
            plt.figure(figsize=(10, 4.0))
            plt.plot(np.nanmean(matrix, axis=0))
            plt.xlabel("Training episode")
            plt.ylabel(key)
            plt.title(f"{framework}: {key}")
            plt.tight_layout()
            plt.show()


## Interpretation checklist

Before reporting an ordering between frameworks, verify that:

- every compared framework has the intended number of completed model seeds;
- evaluation uses the common `eval_episode_seeds` recorded in the JSON;
- confidence intervals are taken from `aggregate[framework][metric]["bootstrap_95_ci"]`;
- quantum and classical parameter counts are reported alongside performance;
- conclusions rely on held-out coverage, coverage AUC, and success—not training return alone.
